# Split cleaned data for the final paper workflow

Workflow:
1. Read `cleaned_data_preprocessing.csv`
2. Split into BEFORE mandate and AFTER mandate using `within_mandate_period`
3. Build 4 tasks:
   - `before_mask`
   - `after_mask`
   - `before_protective`
   - `after_protective`
4. For each task, perform an 80/20 train/test split
5. Save all split datasets and one summary file

Outputs:
`../data/paper_final_splits/`
- `X_train_before_mask.csv`
- `X_test_before_mask.csv`
- `y_train_before_mask.csv`
- `y_test_before_mask.csv`

- `X_train_after_mask.csv`
- `X_test_after_mask.csv`
- `y_train_after_mask.csv`
- `y_test_after_mask.csv`

- `X_train_before_protective.csv`
- `X_test_before_protective.csv`
- `y_train_before_protective.csv`
- `y_test_before_protective.csv`

- `X_train_after_protective.csv`
- `X_test_after_protective.csv`
- `y_train_after_protective.csv`
- `y_test_after_protective.csv`

- `split_summary.csv`

This notebook first sets the input/output paths and the key parameters for splitting. It also defines the required columns that must be present in the cleaned dataset before the workflow can proceed.


In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split


# =========================
# Config
# =========================
INPUT_PATH = Path("../data/cleaned_data_preprocessing.csv")
OUTPUT_DIR = Path("../data/paper_final_splits")

TEST_SIZE = 0.20
RANDOM_STATE = 20240417


# =========================
# Required columns
# =========================
REQUIRED_COLUMNS = [
    "RecordNo",
    "endtime",
    "within_mandate_period",
    "face_mask_behaviour_scale",
    "face_mask_behaviour_binary",
    "protective_behaviour_scale",
    "protective_behaviour_binary",
]


## Helper functions for validation, target normalization, splitting, and saving

This section defines all reusable helper functions used in the workflow.

- `validate_input_columns()` checks whether the cleaned dataset contains all required columns.
- `normalize_period_column()` makes sure `within_mandate_period` is stored as 0/1 only.
- `normalize_binary_target()` converts the target variable into integer binary form.
- `split_task_data()` performs the train/test split, using stratified sampling whenever possible.
- `save_split_files()` saves the generated train/test feature and target files.
- `build_one_task()` wraps the full process for one task, including feature selection, target preparation, splitting, saving, and summary generation.


In [2]:
# =========================
# Helper functions
# =========================
def validate_input_columns(df: pd.DataFrame) -> None:
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(
            "The cleaned dataset is missing required columns:\n"
            + "\n".join(missing)
        )


def normalize_period_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["within_mandate_period"] = df["within_mandate_period"].astype(int)

    valid_values = set(df["within_mandate_period"].dropna().unique().tolist())
    if not valid_values.issubset({0, 1}):
        raise ValueError(
            "Column 'within_mandate_period' must contain only 0 and 1. "
            f"Found values: {sorted(valid_values)}"
        )
    return df


def normalize_binary_target(y: pd.Series) -> pd.Series:
    """
    Convert target to integer 0/1.
    Accepts numeric 0/1, boolean, or strings like '0'/'1'.
    """
    y_clean = y.copy()

    # First try numeric conversion
    try:
        y_num = pd.to_numeric(y_clean, errors="raise").astype(int)
        valid_values = set(y_num.dropna().unique().tolist())
        if valid_values.issubset({0, 1}):
            return y_num.rename("y")
    except Exception:
        pass

    # Then try common string patterns
    mapping = {
        "0": 0,
        "1": 1,
        "false": 0,
        "true": 1,
        "no": 0,
        "yes": 1,
    }

    y_str = y_clean.astype(str).str.strip().str.lower()
    if not set(y_str.unique()).issubset(set(mapping.keys())):
        raise ValueError(
            "Target column could not be safely converted to binary 0/1.\n"
            f"Unique values found: {sorted(y_clean.astype(str).unique().tolist())}"
        )

    return y_str.map(mapping).astype(int).rename("y")


def split_task_data(X: pd.DataFrame, y: pd.Series):
    """
    Use stratified split when both classes exist with enough samples.
    Fall back to unstratified split if necessary.
    """
    class_counts = y.value_counts(dropna=False).to_dict()

    use_stratify = (
        y.nunique() == 2
        and min(class_counts.values()) >= 2
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y if use_stratify else None,
    )

    return X_train, X_test, y_train, y_test


def save_split_files(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    task_name: str,
    output_dir: Path,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    X_train.to_csv(output_dir / f"X_train_{task_name}.csv", index=False)
    X_test.to_csv(output_dir / f"X_test_{task_name}.csv", index=False)

    pd.DataFrame({"y": y_train}).to_csv(
        output_dir / f"y_train_{task_name}.csv",
        index=False
    )
    pd.DataFrame({"y": y_test}).to_csv(
        output_dir / f"y_test_{task_name}.csv",
        index=False
    )


def build_one_task(
    df_subset: pd.DataFrame,
    target_col: str,
    drop_cols: list,
    task_name: str,
    output_dir: Path,
) -> dict:
    missing_drop_cols = [c for c in drop_cols if c not in df_subset.columns]
    if missing_drop_cols:
        # Ignore missing optional columns safely
        drop_cols = [c for c in drop_cols if c in df_subset.columns]

    X = df_subset.drop(columns=drop_cols).copy()
    y = normalize_binary_target(df_subset[target_col])

    if X.empty:
        raise ValueError(f"Task '{task_name}' produced an empty X matrix.")

    X_train, X_test, y_train, y_test = split_task_data(X, y)

    save_split_files(X_train, X_test, y_train, y_test, task_name, output_dir)

    return {
        "task_name": task_name,
        "target": target_col,
        "n_total": len(df_subset),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "n_features": X.shape[1],
        "positive_rate_total": round(float(y.mean()), 6),
        "positive_rate_train": round(float(y_train.mean()), 6),
        "positive_rate_test": round(float(y_test.mean()), 6),
    }


## Main workflow: read data, split by mandate period, build four tasks, and save summary

The final section runs the full workflow.

First, the cleaned dataset is loaded and validated. Then the data is divided into two subsets based on `within_mandate_period`: before mandate and after mandate.

Using these two subsets, four prediction tasks are built:

1. `before_mask`: before mandate data with `face_mask_behaviour_binary` as the target
2. `after_mask`: after mandate data with `face_mask_behaviour_binary` as the target
3. `before_protective`: before mandate data with `protective_behaviour_binary` as the target
4. `after_protective`: after mandate data with `protective_behaviour_binary` as the target

For each task, the code selects features, removes the target-related columns, performs the 80/20 split, saves the outputs, and records summary statistics. At the end, all task summaries are combined into `split_summary.csv`.


In [4]:
# =========================
# Main
# =========================
def main():
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

    df = pd.read_csv(INPUT_PATH, keep_default_na=False)
    validate_input_columns(df)
    df = normalize_period_column(df)

    # Split into before/after mandate
    before_df = df.loc[df["within_mandate_period"] == 0].copy()
    after_df = df.loc[df["within_mandate_period"] == 1].copy()

    if before_df.empty:
        raise ValueError("No rows found for BEFORE mandate data.")
    if after_df.empty:
        raise ValueError("No rows found for AFTER mandate data.")

    summaries = []

    # -----------------------------------
    # Task 1: before mandate + face mask
    # -----------------------------------
    drop_before_mask = [
        "RecordNo",
        "endtime",
        "within_mandate_period",
        "face_mask_behaviour_scale",
        "face_mask_behaviour_binary",
        "protective_behaviour_scale",
        "protective_behaviour_binary",
        # keep protective_behaviour_nomask_scale if it exists
    ]
    summaries.append(
        build_one_task(
            df_subset=before_df,
            target_col="face_mask_behaviour_binary",
            drop_cols=drop_before_mask,
            task_name="before_mask",
            output_dir=OUTPUT_DIR,
        )
    )

    # ----------------------------------
    # Task 2: after mandate + face mask
    # ----------------------------------
    drop_after_mask = [
        "RecordNo",
        "endtime",
        "within_mandate_period",
        "face_mask_behaviour_scale",
        "face_mask_behaviour_binary",
        "protective_behaviour_scale",
        "protective_behaviour_binary",
        # keep protective_behaviour_nomask_scale if it exists
    ]
    summaries.append(
        build_one_task(
            df_subset=after_df,
            target_col="face_mask_behaviour_binary",
            drop_cols=drop_after_mask,
            task_name="after_mask",
            output_dir=OUTPUT_DIR,
        )
    )

    # ------------------------------------------
    # Task 3: before mandate + protective behaviour
    # ------------------------------------------
    drop_before_protective = [
        "RecordNo",
        "endtime",
        "within_mandate_period",
        "face_mask_behaviour_scale",
        "face_mask_behaviour_binary",
        "protective_behaviour_scale",
        "protective_behaviour_binary",
        "protective_behaviour_nomask_scale",  # remove if exists
    ]
    summaries.append(
        build_one_task(
            df_subset=before_df,
            target_col="protective_behaviour_binary",
            drop_cols=drop_before_protective,
            task_name="before_protective",
            output_dir=OUTPUT_DIR,
        )
    )

    # -----------------------------------------
    # Task 4: after mandate + protective behaviour
    # -----------------------------------------
    drop_after_protective = [
        "RecordNo",
        "endtime",
        "within_mandate_period",
        "face_mask_behaviour_scale",
        "face_mask_behaviour_binary",
        "protective_behaviour_scale",
        "protective_behaviour_binary",
        "protective_behaviour_nomask_scale",  # remove if exists
    ]
    summaries.append(
        build_one_task(
            df_subset=after_df,
            target_col="protective_behaviour_binary",
            drop_cols=drop_after_protective,
            task_name="after_protective",
            output_dir=OUTPUT_DIR,
        )
    )

    # Save summary
    summary_df = pd.DataFrame(summaries)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)

    print("\nAll paper-final split files created successfully.\n")
    print("Saved to:", OUTPUT_DIR.resolve())
    print("\nSummary:")
    print(summary_df.to_string(index=False))


if __name__ == "__main__":
    main()



All paper-final split files created successfully.

Saved to: D:\Data Science Project A\All\Data\paper_final_splits

Summary:
        task_name                      target  n_total  n_train  n_test  n_features  positive_rate_total  positive_rate_train  positive_rate_test
      before_mask  face_mask_behaviour_binary    14945    11956    2989          58             0.258347             0.258364            0.258280
       after_mask  face_mask_behaviour_binary    25191    20152    5039          58             0.706482             0.706481            0.706489
before_protective protective_behaviour_binary    14945    11956    2989          57             0.529274             0.529274            0.529274
 after_protective protective_behaviour_binary    25191    20152    5039          57             0.724028             0.724047            0.723953
